### Step 1: Environment Setup & Data Lake Structure
We will install PySpark, create the required directory structure for our Data Lake, and place our raw dataset into the Bronze layer.

In [1]:
# Install PySpark
!pip install pyspark

# Create Data Lake directory structure
import os
import shutil

os.makedirs('data_lake/bronze', exist_ok=True)
os.makedirs('data_lake/silver', exist_ok=True)

# Copy the raw sales.csv into the bronze layer
shutil.copy('/content/sales.csv', 'data_lake/bronze/sales.csv')

# Verify the folder structure
for root, dirs, files in os.walk('data_lake'):
    level = root.replace('data_lake', '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

data_lake/
    bronze/
        sales.csv
    silver/


### Step 2: Load and Inspect with Spark
We will now initialize the SparkSession, load the raw dataset from our Bronze directory, inspect its structure, look at sample records, and analyze potential data quality issues (such as missing values, duplicate entries, invalid values, and format inconsistencies).

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan, trim, lower

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("DataLakePreprocessing") \
    .getOrCreate()

# Load dataset from Bronze layer
bronze_path = "data_lake/bronze/sales.csv"
df_raw = spark.read.csv(bronze_path, header=True, inferSchema=True)

# Display Schema & Total Count
print("=== Dataset Schema ===")
df_raw.printSchema()

total_rows = df_raw.count()
print(f"Total Row Count in Bronze: {total_rows}\n")

# Show a few sample records
print("=== Sample Records ===")
df_raw.show(5)

# Identify exact duplicates
duplicate_count = total_rows - df_raw.dropDuplicates().count()
print(f"Number of exact duplicate rows: {duplicate_count}\n")

# Helper to determine if a column type is numeric (where isnan is valid)
numeric_types = ('double', 'float', 'int', 'integer', 'long')

# Check for Null/NaN values per column safely
print("=== Missing (Null/NaN) Value Counts ===")
null_checks = []
for name, dtype in df_raw.dtypes:
    if dtype in numeric_types:
        condition = col(name).isNull() | isnan(col(name))
    else:
        condition = col(name).isNull()
    null_checks.append(count(when(condition, name)).alias(name))

df_raw.select(null_checks).show()

# Quick analysis of unique string values to identify formatting inconsistencies
text_cols = [c for c, t in df_raw.dtypes if t == 'string']
print("=== Sample Text Field Values (to detect spacing/casing issues) ===")
for c in text_cols[:3]:
    print(f"Distinct values for {c} (truncated):")
    df_raw.select(c).distinct().limit(5).show()

=== Dataset Schema ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

Total Row Count in Bronze: 1000

=== Sample Records ===
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+------

### Step 3: Basic Preprocessing - Silver Layer
Now we perform the preprocessing steps according to the requirements and save the resulting cleaned DataFrame in the Silver folder.

In [19]:
from pyspark.sql.functions import trim, initcap, col, when, date_format, expr

# Configure Spark to maintain standard legacy parser behavior just in case
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# 1. Remove exact duplicate rows
df_silver = df_raw.dropDuplicates()

# 2. Handle missing values
df_silver = df_silver.fillna({
    'customer_name': 'Unknown',
    'payment_method': 'Unknown',
    'city': 'Unknown'
})

# 3. Trim extra spaces and standardize obvious capitalization inconsistencies in text fields
string_columns_to_clean = ['customer_name', 'product', 'category', 'payment_method', 'city', 'state', 'order_status']
for col_name in string_columns_to_clean:
    df_silver = df_silver.withColumn(col_name, initcap(trim(col(col_name))))

# Also clean customer_id specifically
df_silver = df_silver.withColumn('customer_id', trim(col('customer_id')))

# 4. Apply basic validation to quantity, unit price, and discount percentage
df_silver = df_silver.withColumn('quantity', when(col('quantity') <= 0, 1).otherwise(col('quantity'))) \
                     .withColumn('unit_price', when(col('unit_price') < 0, 0).otherwise(col('unit_price'))) \
                     .withColumn('discount_percent', when(col('discount_percent') < 0, 0)
                                                    .when(col('discount_percent') > 100, 100)
                                                    .otherwise(col('discount_percent')))

# 5. Convert valid dates to one consistent format (yyyy-MM-dd) safely.
# We use the SQL-native `try_to_timestamp` which returns NULL on any format or calendar error (such as month 13 or Feb 31) without crashing.
df_silver = df_silver.withColumn('order_date_parsed', expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"))
df_silver = df_silver.withColumn('order_date_parsed',
                                 when(col('order_date_parsed').isNull(), expr("try_to_timestamp(order_date, 'dd-MM-yyyy')"))
                                 .otherwise(col('order_date_parsed')))

# Format successfully parsed dates, and assign "Invalid Date" to anything unparseable safely
df_silver = df_silver.withColumn('order_date',
                                 when(col('order_date_parsed').isNotNull(), date_format(col('order_date_parsed'), 'yyyy-MM-dd'))
                                 .otherwise('Invalid Date')) \
                     .drop('order_date_parsed')

# 6. Save the preprocessed DataFrame inside the Silver folder in CSV format
silver_path = "data_lake/silver/preprocessed_sales"
df_silver.coalesce(1).write.csv(silver_path, header=True, mode="overwrite")

print("Silver Layer DataFrame processed and saved successfully without parsing errors!")

Silver Layer DataFrame processed and saved successfully without parsing errors!


### Step 4: Verification
We show the row counts before and after preprocessing, read the Silver layer data back, and list our applied preprocessing transformations.

In [18]:
# Show the row count before and after preprocessing
print(f"Row count before preprocessing (Bronze): {total_rows}")
silver_count = df_silver.count()
print(f"Row count after preprocessing (Silver): {silver_count}")

# Read the Silver data back with Spark
df_silver_check = spark.read.csv(silver_path, header=True, inferSchema=True)

# Display a few rows of verified silver data
print("\n=== Verification: Sample Silver Layer Records ===")
df_silver_check.show(5)

# Summary of operations performed
print("""
=== Preprocessing Operations Summary ===
1. Removed exact duplicate rows (Dropped 10 duplicate records).
2. Handled missing values by filling Null/NaN values in 'customer_name', 'payment_method', and 'city' with 'Unknown'.
3. Trimmed trailing and leading white spaces from all text/ID fields and applied proper title-casing ('Initcap') to maintain standard consistency.
4. Validated numerical boundaries: handled potential invalid 'quantity' values <= 0 by defaulting to 1, ensured 'unit_price' >= 0, and bounded 'discount_percent' between 0% and 100%.
5. Standardized 'order_date' field into consistent 'yyyy-MM-dd' format and marked clearly invalid calendar dates as 'Invalid Date'.
6. Wrote processed Silver-tier data partition cleanly as CSV to 'data_lake/silver/preprocessed_sales/'.
""")

Row count before preprocessing (Bronze): 1000
Row count after preprocessing (Silver): 990

=== Verification: Sample Silver Layer Records ===
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|     customer_name|             product| category|quantity|unit_price|discount_percent|  payment_method|      city|      state|order_date|order_status|
+--------+-----------+------------------+--------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|  100331|      C0005|      Sakshi Joshi|            Football|   Sports|       2|       670|              30|     Credit Card| Bengaluru|  Karnataka|2025-06-06|   Cancelled|
|  100333|      C0263|Siddharth Kulkarni|    Programming Book|    Books|       4|       670|               0|     Net Banking|      Pune|Maharashtra|2025-05-08|   

In [23]:
import os
import glob
import shutil

# Source folder where Spark wrote the single partitioned CSV file
source_dir = 'data_lake/silver/preprocessed_sales'
target_file = 'data_lake/silver/sales_silver.csv'

# Find the actual CSV file inside the directory (ignoring CRC and SUCCESS files)
csv_files = glob.glob(os.path.join(source_dir, 'part-*.csv'))

if csv_files:
    # Copy the file to the desired destination
    shutil.copy(csv_files[0], target_file)
    print(f'Successfully created single Silver layer CSV at: {target_file}')

    # Display confirmation of size and location
    file_size_kb = os.path.getsize(target_file) / 1024
    print(f'File size: {file_size_kb:.2f} KB')
else:
    print('Error: No processed Spark CSV partition file found.')

Successfully created single Silver layer CSV at: data_lake/silver/sales_silver.csv
File size: 105.06 KB


In [24]:
import os
import shutil

# Creating a simulated 'D:' drive folder in the Colab workspace
d_drive_path = 'D_drive/silver'
os.makedirs(d_drive_path, exist_ok=True)

source_file = 'data_lake/silver/sales_silver.csv'
target_d_file = os.path.join(d_drive_path, 'sales_silver.csv')

if os.path.exists(source_file):
    shutil.copy(source_file, target_d_file)
    print(f'Successfully saved copy to simulated D: Drive path: {target_d_file}')
else:
    print(f'Source file {source_file} not found. Please ensure the previous cells ran successfully.')

Successfully saved copy to simulated D: Drive path: D_drive/silver/sales_silver.csv


In [25]:
from google.colab import files
import os

source_file = 'data_lake/silver/sales_silver.csv'

if os.path.exists(source_file):
    print('Triggering the file download to your local machine...')
    files.download(source_file)
else:
    print(f'Source file {source_file} not found. Please ensure all previous cells ran successfully.')

Triggering the file download to your local machine...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>